Ô 1: cấu hình

In [1]:
from pathlib import Path
import re
import pandas as pd
from datetime import datetime
from collections import Counter

from src.baseline import extract_fields, AMOUNT_RE, DATE_RE

FIELDS = ["company", "date", "address", "total"]
OCR_PATH = Path("outputs/ocr/train_validation_ocr.csv")
GT_TRAIN = Path("outputs/ground_truth/train_ground_truth.csv")
GT_VAL = Path("outputs/ground_truth/validation_ground_truth.csv")
OUT_DIR = Path("outputs/baseline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

Ô 2: kiểm tra dữ liệu OCR

In [2]:
ocr = pd.read_csv(OCR_PATH, dtype={"key": str})
print(ocr.shape, ocr.columns.tolist())
print(ocr["split"].value_counts())
print(ocr["status"].value_counts(dropna=False))
ocr["ocr_text"] = ocr["ocr_text"].fillna("")
print("ocr_text rỗng hoặc dưới 10 ký tự:", (ocr["ocr_text"].str.strip().str.len() < 10).sum())
print("key trùng:", ocr["key"].duplicated().sum())

(626, 4) ['key', 'split', 'ocr_text', 'status']
split
train         501
validation    125
Name: count, dtype: int64
status
success    626
Name: count, dtype: int64
ocr_text rỗng hoặc dưới 10 ký tự: 3
key trùng: 0


Ô 3: đọc nhãn đúng và ghép với OCR

In [3]:
gt = pd.concat([pd.read_csv(GT_TRAIN, dtype=str), pd.read_csv(GT_VAL, dtype=str)])
print(gt.columns.tolist())
need = ["key"] + FIELDS
assert all(c in gt.columns for c in need), f"Thiếu cột, file nhãn đang có: {gt.columns.tolist()}"

gt = gt[need].drop_duplicates("key").rename(columns={f: f"{f}_gold" for f in FIELDS})
df = ocr.merge(gt, on="key", how="left").reset_index(drop=True)
print("Không có nhãn:", df["company_gold"].isna().sum(), "dòng")
df["split"].value_counts()

['key', 'company', 'date', 'address', 'total']
Không có nhãn: 0 dòng


split
train         501
validation    125
Name: count, dtype: int64

Ô 4: hàm chuẩn hóa và đo chỉ số (chỉ dùng để so sánh, giá trị gốc vẫn được lưu nguyên)

In [4]:
DATE_FORMATS = ["%d/%m/%Y", "%d-%m-%Y", "%d.%m.%Y", "%d/%m/%y", "%d-%m-%y", "%d.%m.%y",
                "%Y-%m-%d", "%Y/%m/%d", "%d %b %Y", "%d%b%Y", "%d-%b-%Y", "%d %b %y",
                "%d %B %Y"]

def is_missing(x):
    return x is None or (isinstance(x, float) and pd.isna(x)) or str(x).strip() == ""

def norm_text(s):
    if is_missing(s):
        return None
    s = re.sub(r"[^\w\s]", " ", str(s).lower())
    return re.sub(r"\s+", " ", s).strip()

def norm_total(s):
    if is_missing(s):
        return None
    nums = re.findall(r"\d[\d,.]*", str(s).replace(" ", ""))
    if not nums:
        return None
    tok = nums[-1]
    if re.search(r",\d{2}$", tok):          # 60,34 -> 60.34
        tok = tok[:-3] + "." + tok[-2:]
    tok = tok.replace(",", "")
    try:
        return f"{float(tok):.2f}"
    except ValueError:
        return None

def norm_date(s):
    if is_missing(s):
        return None
    core = re.sub(r"\s*([/\-.])\s*", r"\1", str(s).strip())
    for cand in (core, core.replace(" ", "")):
        for f in DATE_FORMATS:
            try:
                return datetime.strptime(cand, f).strftime("%Y-%m-%d")
            except ValueError:
                pass
    return norm_text(core)

NORM = {"company": norm_text, "address": norm_text, "date": norm_date, "total": norm_total}

def exact(pred, gold, f):
    if is_missing(pred) or is_missing(gold):
        return 0
    return int(NORM[f](pred) == NORM[f](gold))

def token_f1(pred, gold):
    p = (norm_text(pred) or "").split()
    g = (norm_text(gold) or "").split()
    if not p or not g:
        return 0.0
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0:
        return 0.0
    pr, rc = common / len(p), common / len(g)
    return 2 * pr * rc / (pr + rc)

Ô 5: chạy baseline trên train và validation

In [5]:
preds = df["ocr_text"].apply(extract_fields)
res = df[["key", "split"]].copy()
for f in FIELDS:
    res[f"{f}_pred"] = [p[f] for p in preds]
    res[f"{f}_gold"] = df[f"{f}_gold"]
    res[f"{f}_em"] = [exact(p, g, f) for p, g in zip(res[f"{f}_pred"], res[f"{f}_gold"])]
res["company_f1"] = [token_f1(p, g) for p, g in zip(res["company_pred"], res["company_gold"])]
res["address_f1"] = [token_f1(p, g) for p, g in zip(res["address_pred"], res["address_gold"])]
res.head()

,key,split,company_pred,company_gold,company_em,date_pred,date_gold,date_em,address_pred,address_gold,address_em,total_pred,total_gold,total_em,company_f1,address_f1
0,X00016469612,train,tan woon yann,BOOK TA .K (TAMAN DAYA) SDN BHD,0,25/12/2018,25/12/2018,1,BOOK TA -K (TAMAN DAYA) SDN BHD NO.S: $557 & $...,"NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 8...",0,91.00,9.00,0,0.0,0.588235
1,X00016469619,train,tan woon yann,INDAH GIFT & HOME DECO,0,19/10/2018,19/10/2018,1,"27, JALAN DEDAF 13, TAMAN JOHOR JAYA, 81100 JO...","27, JALAN DEDAP 13, TAMAN JOHOR JAYA, 81100 JO...",0,"60,34",60.30,0,0.0,0.909091
2,X00016469620,train,tan woon yann,MR D.I.Y. (JOHOR) SDN BHD,0,12-01-19,12-01-19,1,"MR O.1.Y. (JOHOR) SDN BHD LOT 1851-A & 1851-B,...","LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERI...",0,"33,90",33.90,1,0.0,0.780488
3,X00016469622,train,tan woon yann,YONGFATT ENTERPRISE,0,25/12/2018,25/12/2018,1,NO.122.124. JALAN DEDAP 13 81100 JOHOR BAHRU,NO 122.124. JALAN DEDAP 13 81100 JOHOR BAHRU,1,80.90,80.90,1,0.0,1.000000
4,X00016469623,train,tan woon yann,MR D.I.Y. (M) SDN BHD,0,4 - 6/300,18-11-18,0,"{07 185)-A & 1851-B, JALAN KPB 6, KAWASAN PERI...","LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERI...",0,30.90,30.90,1,0.0,0.727273


Ô 6: bảng chỉ số

In [6]:
def summary(d):
    s = {f: d[f"{f}_em"].mean() for f in FIELDS}
    s["TRUNG_BINH"] = sum(s.values()) / 4
    s["company_F1"] = d["company_f1"].mean()
    s["address_F1"] = d["address_f1"].mean()
    for f in FIELDS:
        s[f"{f}_None"] = d[f"{f}_pred"].isna().mean()   # tỉ lệ không tìm thấy
    return pd.Series(s).round(3)

table = pd.DataFrame({sp: summary(res[res["split"] == sp]) for sp in ["train", "validation"]})
table

,train,validation
company,0.255,0.248
date,0.695,0.752
address,0.253,0.304
total,0.533,0.568
TRUNG_BINH,0.434,0.468
company_F1,0.514,0.498
address_F1,0.700,0.757
company_None,0.008,0.016
date_None,0.120,0.096
address_None,0.132,0.072


Ô 7: phân tích lỗi, tách “đọc chữ” và “tìm trường”

In [7]:
def gold_in_ocr(f, gold, text):
    """Thông tin đúng có nằm trong chữ OCR không?"""
    if is_missing(gold):
        return False
    if f == "total":
        return norm_total(gold) in {norm_total(a) for a in AMOUNT_RE.findall(text)}
    if f == "date":
        return norm_date(gold) in {norm_date(m.group(0)) for m in DATE_RE.finditer(text)}
    g = set((norm_text(gold) or "").split())
    t = set((norm_text(text) or "").split())
    return len(g) > 0 and len(g & t) / len(g) >= 0.8

for f in FIELDS:
    causes = []
    for i, r in res.iterrows():
        if r[f"{f}_em"] == 1:
            causes.append("dung")
        elif gold_in_ocr(f, r[f"{f}_gold"], df.loc[i, "ocr_text"]):
            causes.append("loi_tim_truong")     # OCR có chữ đúng nhưng luật chọn sai
        else:
            causes.append("loi_doc_chu")        # chữ đúng không có trong OCR
    res[f"{f}_loi"] = causes

err = pd.DataFrame({f: res[res["split"] == "validation"][f"{f}_loi"].value_counts() for f in FIELDS}).fillna(0).astype(int)
err

,company,date,address,total
dung,31,94,38,71
loi_doc_chu,27,22,20,14
loi_tim_truong,67,9,67,40


Ô code xem chữ OCR của các lỗi (để lấy ≥ 3 lỗi thật cho báo cáo)

In [ ]:
def show_error(field, n=3, split="validation", cause="loi_tim_truong"):
    sub = res[(res["split"] == split) & (res[f"{field}_loi"] == cause)].head(n)
    for i, r in sub.iterrows():
        print("=" * 70)
        print("KEY:", r["key"], "| trường:", field, "| loại:", cause)
        print("DỰ ĐOÁN:", r[f"{field}_pred"])
        print("ĐÁP ÁN :", r[f"{field}_gold"])
        print("--- OCR ---")
        print(df.loc[i, "ocr_text"][:900])

show_error("total", 3, cause="loi_tim_truong")

Xem một số lỗi thật của validation để ghi vào báo cáo:

In [8]:
f = "total"   # đổi thành company / date / address
v = res[(res["split"] == "validation") & (res[f"{f}_em"] == 0)]
v[["key", f"{f}_pred", f"{f}_gold", f"{f}_loi"]].head(15)

,key,total_pred,total_gold,total_loi
14,X51005268472,0.00,32.70,loi_tim_truong
20,X51005337872,64.50,54.50,loi_doc_chu
29,X51005365179,20.00,21.20,loi_tim_truong
40,X51005441401,None,343.95,loi_doc_chu
46,X51005442338,None,28.60,loi_tim_truong
61,X51005444037,None,129.30,loi_tim_truong
77,X51005568827,23.26,23.25,loi_tim_truong
91,X51005587267,None,85.10,loi_tim_truong
98,X51005663272,None,42.40,loi_tim_truong
115,X51005676539,22.00,23.32,loi_tim_truong


In [ ]:
Ô 8: lưu kết quả

In [9]:
cols = (["key", "split"]
        + [f"{f}_pred" for f in FIELDS] + [f"{f}_gold" for f in FIELDS]
        + [f"{f}_em" for f in FIELDS] + ["company_f1", "address_f1"]
        + [f"{f}_loi" for f in FIELDS])
res[res["split"] == "train"][cols].to_csv(OUT_DIR / "baseline_train.csv", index=False)
res[res["split"] == "validation"][cols].to_csv(OUT_DIR / "baseline_validation.csv", index=False)
print("Đã lưu vào", OUT_DIR)

Đã lưu vào outputs\baseline


## Kết quả baseline (validation, 125 hóa đơn)
Exact match: company 0.248, date 0.752, address 0.304, total 0.568, trung bình 0.468.
F1 token: company 0.498, address 0.757.

## Lỗi thường gặp (bàn giao người 4)
1. company, address: 67/125 hóa đơn mỗi trường là lỗi tìm trường, tức OCR có chữ đúng nhưng luật chọn sai dòng (vd dòng đầu là tên người, địa chỉ thiếu hoặc thừa dòng).
2. total: 40 lỗi tìm trường (lấy nhầm số cuối dòng của bảng GST Summary, bỏ sót từ khóa dính liền như "NetTotal"), 14 lỗi đọc chữ (OCR đọc sai chữ số).
3. date: chủ yếu lỗi đọc chữ (22 lỗi), chỉ 9 lỗi tìm trường. Cần làm sạch ảnh hoặc sửa ký tự.
4. Khoảng 10% date và 15% total không tìm thấy (None).
5. Gợi ý cải tiến: so khớp từ khóa bỏ khoảng trắng, bỏ dòng GST Summary khi tìm total, dùng vị trí dòng cho company/address, tiền xử lý ảnh trước OCR.